# F5-TTS v1 Base — English voice test
Run in Google Colab. Select a GPU runtime if available. This notebook installs the official F5-TTS package and generates speech conditioned on a voice clip you upload and its transcript.


In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install f5-tts


In [ ]:
import torch, os
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
else: print('No GPU assigned. You can try CPU, but generation may be very slow or run out of memory.')


## Upload a reference voice clip
Use a clean WAV recording (roughly 5–15 seconds) and type exactly what is spoken in it. Only use a voice you have permission to use.


In [ ]:
from google.colab import files
uploaded = files.upload()
ref_file = next(iter(uploaded))
ref_text = input('Exact transcript of the uploaded reference audio: ').strip()
assert ref_text, 'Reference transcript is required.'


## Generate the shared comparison sample
The same sample sentence is used in both model notebooks.


In [ ]:
from f5_tts.api import F5TTS
import os
tts = F5TTS(model='F5TTS_v1_Base', device='cuda' if torch.cuda.is_available() else 'cpu')
sample_text = 'Welcome to the voice comparison. This is a natural English speech sample generated with an open source text to speech model. The goal is clear pronunciation, smooth pacing, and a confident, conversational delivery.'
wav, sample_rate, spectrogram = tts.infer(
    ref_file=ref_file,
    ref_text=ref_text,
    gen_text=sample_text,
    file_wave='f5_tts_english_sample.wav',
)
print('Saved f5_tts_english_sample.wav at', sample_rate, 'Hz')


In [ ]:
from IPython.display import Audio, display
display(Audio('f5_tts_english_sample.wav'))
!ffmpeg -y -loglevel error -i f5_tts_english_sample.wav -codec:a libmp3lame -qscale:a 3 f5_tts_english_sample.mp3
files.download('f5_tts_english_sample.mp3')
